In [4]:
import os
import pandas as pd
import requests
import time
import json
import hashlib
from tqdm import tqdm
from dotenv import load_dotenv
import requests

from professions import it_vacancies_dict

pd.set_option('display.max_columns', None)

### Работа.ру

In [19]:
def get_signature() -> str:
    """
    Точная реализация подписи из документации Rabota.ru
    Аналог PHP функции getSignature()
    """
    secret = "6q1Fqqf1ACSLplIGlNti21fIISfIqwMF"
    params = {
        "app_id": "869",
        "time": int(time.time()),
        "code": "64N1JCqShvAgeqx7qYnaHO10Qd6W2kOq",
    }
    # 1. Приводим все значения к строке (как в PHP)
    params = {k: str(v) for k, v in params.items()}
    
    # 2. Рекурсивная сортировка по ключам (ksort)
    def sort_recursive(arr):
        if not isinstance(arr, dict):
            return arr
        sorted_arr = {k: sort_recursive(arr[k]) for k in sorted(arr.keys())}
        return sorted_arr
    
    sorted_params = sort_recursive(params)
    
    # 3. Преобразуем в JSON без пробелов (как json_encode без флагов)
    json_str = json.dumps(sorted_params, separators=(',', ':'))
    
    # 4. Добавляем секрет и вычисляем SHA256
    signature_string = json_str + secret
    signature = hashlib.sha256(signature_string.encode()).hexdigest()
    
    return signature

In [21]:
url = "https://api.rabota.ru/oauth/token.json"
headers = {
    "content-type": "application/x-www-form-urlencoded"
}
data = {
    "app_id": 869,
    "time": int(time.time()),
    "code": "64N1JCqShvAgeqx7qYnaHO10Qd6W2kOq",
    "signature": get_signature()
}

response = requests.post(url, headers=headers, data=data)
print(f"Status code: {response.status_code}")
print(f"Response: {response.text}")

Status code: 200
Response: {"access_token":"bM60pKJE2vMakrOtE2itdhr91gf4q2v2","expires_in":86400}


In [41]:
load_dotenv()

APP_ID = os.getenv("APP_ID_RABOTARU")
ACCESS_TOKEN = os.getenv("X_TOKEN_RABOTARU")

# Заголовки, обязательные для API Rabota.ru
headers = {
    "x-token": ACCESS_TOKEN,
    "application-id": APP_ID,
    "Content-Type": "application/json"
}

json_vacancies = {
    'vacancy_ids': [54260512]
}


# Попробуем получить список вакансий (эндпоинт может отличаться)
# Сначала проверим, что токен работает — сделаем запрос к справочнику профессий
url = "https://api.rabota.ru/v4/vacancies.json"

response = requests.post(url, headers=headers, json=json_vacancies)

print(f"Статус: {response.status_code}")
if response.status_code == 200:
    print("✅ API работает, токен валиден!")
    data = response.json()
else:
    print(f"❌ Ошибка: {response.text}")

Статус: 400
❌ Ошибка: {"errors":[{"code":"FIELD_REQUIRED","field":"request.vacancy_ids","is_system":true,"user_message":"Серверная ошибка","system_message":"Отсутствует обязательное поле"}]}


### HH

In [2]:
import hhru

client = hhru.Client()

vacancies = client.search_vacancies_over_pages(
                text='CV engineer',
                search_field="name"
            )
vacancies_list = list(map(lambda x: x.model_dump(), vacancies))
print(f'Найдено вакансий CV engineer -> {len(vacancies_list)}')

Exception: {'errors': [{'type': 'forbidden'}], 'request_id': '1776613917161409c84dbac605889022'}

In [5]:
def search_hh_vacancies(query, pages=5):
    """
    Поиск вакансий на hh.ru с обходом 403
    """
    # Создаем сессию с ретраями
    session = requests.Session()
    retries = Retry(total=3, backoff_factor=1, status_forcelist=[403, 500, 502, 503, 504])
    session.mount('https://', HTTPAdapter(max_retries=retries))
    
    # Полные заголовки как у реального браузера
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json',
        'Accept-Language': 'ru-RU,ru;q=0.9,en;q=0.8',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Referer': 'https://hh.ru/',
        'Sec-Fetch-Dest': 'empty',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-site'
    }
    
    all_vacancies = []
    
    for page in range(pages):
        params = {
            'text': query,
            'search_field': 'name',
            'page': page,
            'per_page': 100
        }
        
        try:
            response = session.get(
                'https://api.hh.ru/vacancies',
                params=params,
                headers=headers,
                timeout=30
            )
            
            if response.status_code == 200:
                data = response.json()
                items = data.get('items', [])
                all_vacancies.extend(items)
                print(f'Страница {page}: {len(items)} вакансий')
                
                if page >= data.get('pages', 1) - 1:
                    break
                    
                time.sleep(0.5)
                
            elif response.status_code == 403:
                print(f'Ошибка 403. Пробую другой User-Agent...')
                # Пробуем другой User-Agent
                headers['User-Agent'] = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
                continue
            else:
                print(f'Ошибка {response.status_code}')
                break
                
        except Exception as e:
            print(f'Исключение: {e}')
            break
    
    return all_vacancies

# Использование
vacancies = search_hh_vacancies('CV engineer')
print(f'\nВсего найдено: {len(vacancies)}')

NameError: name 'Retry' is not defined

In [27]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

response = requests.get(
    'https://api.hh.ru/vacancies',
    params={'text': 'CV engineer', 'search_field': 'name', 'per_page': 10},
    headers=headers
)

print(response.status_code)

403


In [ ]:
df = pd.DataFrame(vacancies_list)
df.to_parquet(f'data/Искусственный интеллект/CV engineer.parquet')

In [86]:
for spec in it_vacancies_dict:
    spec_folder = f'data/{spec}'
    os.makedirs(spec_folder, exist_ok=True)
    
    for vacancy_name in tqdm(it_vacancies_dict[spec], desc=f'{spec}'):
            vacancies = client.search_vacancies_over_pages(
                text=vacancy_name,
                search_field="name"
            )
            vacancies_list = list(map(lambda x: x.model_dump(), vacancies))
            print(f'Найдено вакансий {vacancy_name} -> {len(vacancies_list)}')

            df = pd.DataFrame(vacancies_list)
            df.to_parquet(f'data/{spec}/{vacancy_name}.parquet')

Аналитик:   6%|▋         | 1/16 [00:05<01:22,  5.51s/it]

Найдено вакансий Аналитик данных -> 562


Аналитик:  12%|█▎        | 2/16 [00:07<00:51,  3.68s/it]

Найдено вакансий Data Analyst -> 219


Аналитик:  19%|█▉        | 3/16 [00:09<00:37,  2.87s/it]

...

Найдено вакансий Smart Contract Developer -> 55


Новые профессии:  94%|█████████▍| 15/16 [00:10<00:00,  1.36it/s]

Найдено вакансий DeFi специалист -> 0


Новые профессии: 100%|██████████| 16/16 [00:11<00:00,  1.43it/s]

Найдено вакансий Metaverse Developer -> 0


In [ ]:
for spec in os.listdir("data"):
    spec_path = f"data/{spec}"
    if not os.path.isdir(spec_path):
        continue
    
    print(f"\n📁 {spec}")
    
    # Собираем все DataFrame из parquet файлов
    all_dfs = []
    
    for file in tqdm(os.listdir(spec_path)):
        if not file.endswith('.parquet'):
            continue
            
        # Читаем файл
        df = pd.read_parquet(f"{spec_path}/{file}")
        
        # Добавляем колонку с тегом (имя файла)
        df['search_tag'] = file.replace('.parquet', '')
        
        all_dfs.append(df)
    
    if all_dfs:
        result = pd.concat(all_dfs, ignore_index=True)
        result = result.drop_duplicates('alternate_url').reset_index(drop=True)
        # Приводим все колонки к простым типам
        for col in result.columns:
            # Преобразуем boolean в int (True/False → 1/0)
            if result[col].dtype == 'bool':
                result[col] = result[col].astype('int8')
            # Преобразуем mixed типы в строки
            elif result[col].dtype == 'object':
                result[col] = result[col].astype('string')
        result['id'] = result['id'].astype(int)
        # Сохраняем как CSV (надёжнее, чем parquet для смешанных типов)
        result.to_parquet(f"dataset/{spec}.parquet", index=False)
        print(f"   ✅ Сохранено: dataset/{spec}.parquet ({len(result)} вакансий)")


📁 Data Engineering


100%|██████████| 12/12 [00:00<00:00, 133.89it/s]


   ✅ Сохранено: dataset/Data Engineering.parquet (594 вакансий)

📁 DevOps и инфраструктура


100%|██████████| 29/29 [00:01<00:00, 25.87it/s]


   ✅ Сохранено: dataset/DevOps и инфраструктура.parquet (5136 вакансий)

📁 Аналитик


...

   ✅ Сохранено: dataset/Разработчик.parquet (6383 вакансий)

📁 Тестировщик


100%|██████████| 15/15 [00:00<00:00, 24.46it/s]


   ✅ Сохранено: dataset/Тестировщик.parquet (3844 вакансий)

📁 Технический специалист


100%|██████████| 17/17 [00:00<00:00, 27.38it/s]


   ✅ Сохранено: dataset/Технический специалист.parquet (2806 вакансий)


In [120]:
for spec in tqdm(os.listdir("dataset")):
    if not spec.endswith('.parquet'):
        continue
    
    filepath = f"dataset/{spec}"
    df = pd.read_parquet(filepath)
    
    print(f"\n{spec}: {len(df)} вакансий")
    
    for i, row in df.iterrows():
        vacancy_id = row['id']
        
        try:
            url = f"https://api.hh.ru/vacancies/{vacancy_id}"
            response = requests.get(url, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                df.at[i, 'description'] = data.get('description', '')
                
                skills = [s['name'] for s in data.get('key_skills', [])]
                df.at[i, 'key_skills'] = str(skills)
            
        except Exception as e:
            print(f"Ошибка {vacancy_id}: {e}")

        time.sleep(0.2)
    
    # Сохраняем как parquet (поддерживает списки)
    df.to_parquet(filepath, index=False)
    print(f"✅ {spec} сохранён как parquet")

  0%|          | 0/10 [00:00<?, ?it/s]


Data Engineering.parquet: 594 вакансий


 10%|█         | 1/10 [08:40<1:18:05, 520.65s/it]

✅ Data Engineering.parquet сохранён как parquet

DevOps и инфраструктура.parquet: 5136 вакансий
Ошибка 131708151: HTTPSConnectionPool(host='api.hh.ru', port=443): Read timed out. (read timeout=10)
Ошибка 131790103: HTTPSConnectionPool(host='api.hh.ru', port=443): Read timed out. (read timeout=10)
Ошибка 131589183: HTTPSConnectionPool(host='api.hh.ru', port=443): Read timed out. (read timeout=10)


 20%|██        | 2/10 [1:21:49<6:12:47, 2795.91s/it]

...

✅ Разработчик.parquet сохранён как parquet

Тестировщик.parquet: 3844 вакансий


 90%|█████████ | 9/10 [7:07:24<50:14, 3014.11s/it]  

✅ Тестировщик.parquet сохранён как parquet

Технический специалист.parquet: 2806 вакансий


100%|██████████| 10/10 [7:40:52<00:00, 2765.21s/it]

✅ Технический специалист.parquet сохранён как parquet


In [16]:
headers = {
    'User-Agent': 'My app/1.1 (contact: aleksandrsarajkin95@gmail.com)',
    'Accept': 'application/json'
}

try:
    url = f"https://api.hh.ru/vacancies/132179397"
    response = requests.get(url, headers=headers, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        print(data)
        # df.at[i, 'description'] = data.get('description', '')
        
        skills = [s['name'] for s in data.get('key_skills', [])]
        # df.at[i, 'key_skills'] = str(skills)
    else:
        print(response.status_code)
    
except Exception as e:
    print(f"Ошибка: {e}")

403
